In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
# adata = ad.read_h5ad("./data/larry/postprocessed.h5ad")
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

# Step 1 

## marker boosting

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# 1. Get the matrix
# scVelo and Scanpy usually run PCA on log-transformed spliced counts
X = adata.layers["spliced"].copy()

# 2. Subset to highly variable genes
hvg_mask = adata.var["highly_variable"].values
X = X[:, hvg_mask]

# 3. Log1p transform (Scanpy's default)
X = X.toarray() if hasattr(X, "toarray") else X
X = np.log1p(X)

# 4. Center and scale (Scanpy centers but doesn’t always scale variance)
scaler = StandardScaler(with_mean=True, with_std=True)
X = scaler.fit_transform(X)

marker_dict = {
    "Mast": ["Cma1", "Tph1", "Papss2", "Fcer1a", "Gzmb"],
    "Basophil": ["Hgf", "Ccl3", "Slpi"],
    "Eosinophil": ["Prg3", "Epx", "Prg2"],
    "Megakaryocyte": ["Ppbp", "Thbs1", "Pf4", "Timp3"],
    "Monocyte": ["Ms4a6d", "Fabp5", "Ctss", "Ms4a6c", "Tgfbi",
                 "Olfm1", "Csf1r", "Ccr2", "Klf4", "F13a1"],
    "Neutrophil": ["S100a9", "Itgb2l", "Elane", "Fcnb",
                   "Mpo", "Prtn3", "S100a6", "S100a8",
                   "Lcn2", "Lrg1"],
    "Lymphoid": ["Ighm", "Satb1", "Dntt", "Ctr9", "Jchain"],
    "migDC": ["H2-Eb1", "H2-Aa", "H2-Ab1", "H2-DMb1", "Ccr7"],
    "pDC": ["Siglech"],
    "Erythroid": ["Hbb-bs"],
    "cDC": ["Cst3", "Xcr1"]
}

booster_dict = {
    "Mast": 1.0,
    "Basophil": 1.0,
    "Eosinophil": 1.0,
    "Megakaryocyte": 1.0,
    "Monocyte": 1.0,      # no boost
    "Neutrophil": 1.0,
    "Lymphoid": 1.0,
    "migDC": 1.0,
    "pDC": 1.0,
    "Erythroid": 1.0,
    "cDC": 1.0
}

# Copy to avoid side-effects
X_boosted = X.copy()

# Loop over cell types in booster_dict
for celltype, genes in marker_dict.items():
    boost = booster_dict.get(celltype, 1.0)
    if boost == 1.0:
        continue
    
    # find indices of marker genes among HVGs
    idx = [i for i, g in enumerate(adata.var_names[hvg_mask]) if g in genes]
    if idx:
        print(f"Boosting {celltype} markers ({len(idx)} genes) by {boost}x")
        X_boosted[:, idx] *= boost


from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import *
from scipy.sparse import issparse

# ---------- 1. Load full dataset ----------

from sklearn.preprocessing import StandardScaler

# --- extract and subset ---
V = adata.layers["velocity"]  # or "velocity_pyro"
V = V[:, hvg_mask]
V = V.toarray() if hasattr(V, "toarray") else V

# --- compute per-gene mean, safely ---
gene_means = np.nanmean(V, axis=0)

# replace genes where mean is NaN (all-NaN columns)
nan_gene_mask = np.isnan(gene_means)
gene_means[nan_gene_mask] = 0.0  # set missing genes to 0

# --- impute ---
inds = np.where(np.isnan(V))
V[inds] = np.take(gene_means, inds[1])

# --- sanity check ---
print(f"Total NaN after imputation: {np.isnan(V).sum()}")

# --- scale (variance = 1 per gene) ---
scaler_v = StandardScaler(with_mean=False, with_std=True)
V = scaler_v.fit_transform(V)


# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="euclidean",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
#     alpha=0.5,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=200,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

# plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["time_info"].astype(str).values),  # ✅ categorical strings
    grid_density=1,
    stream_density=1.2,
    scatter_size=10,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    cmap="tab10",      # tab10 gives 10 distinct discrete colors
    grid_size=30,
    show_labels=False
)

# Phase distance vs Euclidean distance

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=200,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.1,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.25,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.5,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=1,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["time_info"].astype(str).values),  # ✅ categorical strings
    grid_density=1,
    stream_density=1.2,
    scatter_size=10,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    cmap="viridis",      # tab10 gives 10 distinct discrete colors
    grid_size=30,
    show_labels=False
)

# Step 3
## Select knn

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=15              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=20              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=50              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values))

# Step 5
## PCA

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=20,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=40,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=50,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

# min_dist

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.1
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.3
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.7
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.9
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_2d(emb.X_emb, list(adata.obs["state_info"].values), alpha=0.3)

# Seed

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=11)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["time_info"].astype(str).values),  # ✅ categorical strings
    grid_density=1,
    stream_density=1.2,
    scatter_size=10,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    cmap="tab10",      # tab10 gives 10 distinct discrete colors
    grid_size=30,
    show_labels=False
)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=42)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["time_info"].astype(str).values),  # ✅ categorical strings
    grid_density=1,
    stream_density=1.2,
    scatter_size=10,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    cmap="tab10",      # tab10 gives 10 distinct discrete colors
    grid_size=30,
    show_labels=False
)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=77)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["time_info"].astype(str).values),  # ✅ categorical strings
    grid_density=1,
    stream_density=1.2,
    scatter_size=10,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    cmap="tab10",      # tab10 gives 10 distinct discrete colors
    grid_size=30,
    show_labels=False
)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=101)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["time_info"].astype(str).values),  # ✅ categorical strings
    grid_density=1,
    stream_density=1.2,
    scatter_size=10,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    cmap="tab10",      # tab10 gives 10 distinct discrete colors
    grid_size=30,
    show_labels=False
)

In [ ]:
# UMAP parameters
umap_params = {
    "min_dist": 0.5
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X_boosted, V,
    dist_method="phase",
    dof=50,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0.75,
    pca_components=30,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding(seed=123)

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=0.8,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["time_info"].astype(str).values),  # ✅ categorical strings
    grid_density=1,
    stream_density=1.2,
    scatter_size=10,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    cmap="tab10",      # tab10 gives 10 distinct discrete colors
    grid_size=30,
    show_labels=False
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.neighbors import NearestNeighbors
from scripts.plotting import compute_velocity_on_grid

# --- helper: trim seed points outside the manifold --------------------------
def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb, n_neighbors=k)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


def add_manual_arrows(ax, coords, X_emb, V_pred, color="k", scale=2.0, width=0.004):
    """
    Add quiver arrows at given coordinates by snapping to nearest cell
    and using its velocity vector.
    """
    nn = NearestNeighbors(n_neighbors=1).fit(X_emb)
    _, idx = nn.kneighbors(coords)
    idx = idx.ravel()

    ax.quiver(
        X_emb[idx, 0], X_emb[idx, 1],
        V_pred[idx, 0], V_pred[idx, 1],
        angles="xy", scale_units="xy", scale=3,
        width=0.003,
        headwidth=4.5, headlength=4.0, headaxislength=2.3,  # smaller head
        minlength=0.2,    # ensure shaft is drawn
        color="k", alpha=0.9
    )
    return ax


# === 1) Embedding and labels ================================================
X_emb = emb.X_emb
labels = np.asarray(adata.obs["state_info"].values)

# === 2) Colors (Undifferentiated grey, others tab10) ========================
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"
cell_colors = np.array([colmap[lab] for lab in labels])

# === 3) Grid seeds (mass filter + boundary trim) ============================
Xg, keep_mass, Vg = compute_velocity_on_grid(X_emb, grid_size=25, min_mass=0.01)
keep_inside = points_inside_mask(X_emb, Xg, k=8, radius_scale=1.2)
Xg = Xg[keep_inside]
Vg = emb.tps_vf.predict(Xg)

# === 4) Plot ================================================================
fig, ax = plt.subplots(figsize=(12, 12))

# Scatter: faded "Undifferentiated" background
if "Undifferentiated" in uniq:
    undiff = labels == "Undifferentiated"
    ax.scatter(X_emb[undiff, 0], X_emb[undiff, 1],
               c="#d3d3d3", s=40, alpha=0.2, linewidths=0)

# Scatter: all other cells
mask = labels != "Undifferentiated"
ax.scatter(X_emb[mask, 0], X_emb[mask, 1],
           c=cell_colors[mask], s=80, alpha=0.4, linewidths=0)

# Quiver: trimmed seeds, styled arrows
ax.quiver(
    Xg[:, 0], Xg[:, 1], Vg[:, 0], Vg[:, 1],
    angles="xy", scale_units="xy", scale=3,
    width=0.003,
    headwidth=4.5, headlength=4.0, headaxislength=2.3,  # smaller head
    minlength=0.2,    # ensure shaft is drawn
    color="k", alpha=0.9
)



# # Example patch coordinates
patch_coords = [[12.0,10.4],
    [14,4],
    [13,4]]

# # Predict velocity at *all cells*
V_cells = emb.tps_vf.predict(X_emb)

# # Add patched arrows
add_manual_arrows(ax, patch_coords, X_emb, V_cells, color="k", scale=2.0, width=0.004)

# Legend (skip Undifferentiated for clarity)
handles = [plt.Line2D([], [], marker='o', linestyle='',
                      color=colmap[lab], label=lab, markersize=5)
           for lab in other]
# ax.legend(handles=handles, fontsize=8, loc='center left',
#           bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)

# Clean axes
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("./figures/larry_velocity_umap_large.pdf", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# === 2) Legend-only plot (includes Undifferentiated) ========================
fig, ax = plt.subplots(figsize=(4, 6))

handles = [plt.Line2D([], [], marker='o', linestyle='',
                      color=colmap[lab], label=lab, markersize=8)
           for lab in uniq]  # <-- now includes "Undifferentiated"

ax.legend(handles=handles, fontsize=10, loc='center')
ax.axis("off")

plt.tight_layout()
plt.savefig("./figures/larry_velocity_umap_legend.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Example patch coordinates (replace with your list)
# patch_coords = [
#     [11.8,10.2],
#     [14,4],
#     [13,4],
#     [-0.661,  7.997]
# ]

patch_coords = [
    [8,18],
    [12,2],
    [5.8,-1.5],
    [-2.7,-3],
    [-3.3,3.6],
    [-4.5,7],
    [-1.5,8.5],
    [0,12]
]

fig, ax = plt.subplots(figsize=(10, 10))

# Scatter all cells
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=cell_colors, s=30, alpha=0.4, linewidths=0
)

# Overlay patch coordinates
patch_coords = np.array(patch_coords)
ax.scatter(
    patch_coords[:, 0], patch_coords[:, 1],
    c="red", s=80, marker="x", label="Patch coords"
)

# Equal aspect
ax.set_aspect("equal")

# Dense grid
ax.grid(True, linestyle="--", alpha=0.6, color="grey")
ax.xaxis.set_major_locator(MultipleLocator(1))   # grid every 1 unit
ax.yaxis.set_major_locator(MultipleLocator(1))   # grid every 1 unit
ax.minorticks_on()                               # finer ticks
ax.xaxis.set_minor_locator(MultipleLocator(0.5))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))
ax.grid(which="minor", linestyle=":", alpha=0.3)

# Legend
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
import joblib
import numpy as np

# 1) Save embedding coordinates
np.save("./data/larry/larry_umap_embedding.npy", emb.X_emb)

# 2) Save the entire embedder object (pickle via joblib)
joblib.dump(emb, "./data/larry/larry_embedder.pkl")

print("Saved: embedding and emb object 🎉")


In [ ]:
import joblib
import pandas as pd

# Load the saved embedder object
emb = joblib.load("./data/larry/larry_embedder.pkl")

# Extract the embedding coordinates
X_emb = emb.X_emb  # shape (N, 2)

# Save as CSV
pd.DataFrame(X_emb, columns=["dim1", "dim2"]).to_csv(
    "./data/larry/larry_flowmap_embedding.csv",
    index=False
)

print("Saved: larry_umap_embedding.csv 🎉")